# Node 4 — Duplicate Control playground

Demonstrates `DuplicateControlNode` end-to-end:
SHA-256 exact-match check → cosine-similarity near-duplicate detection → SSE events → audit record.

Node 4 always runs **after** Node 3: it receives `sha256` (from Node 1) and `text`
(extracted content) and rejects documents that are exact or semantic duplicates of
previously ingested ones.

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [1]:
import asyncio
import hashlib

from classiflow.shared.audit.service import AuditService
from classiflow.shared.database.base import Base
from classiflow.shared.database.repositories.audit import SqlAuditRepository
from classiflow.shared.database.repositories.hash import SqlHashRepository
from classiflow.shared.events.broadcaster import EventBroadcaster
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from classiflow.ingesta.nodes.node4_duplicate_control import DuplicateControlNode, EmbeddingStore

print("imports OK")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


## 2 — Database setup

Creates the SQLite engine and ensures all tables exist.
The `session_factory` is reused by every section below.

In [2]:
from pathlib import Path

import classiflow.settings as _settings_mod

# settings.py lives at src/classiflow/settings.py → parents[2] = project root
_project_root = Path(_settings_mod.__file__).parents[2]
_db_path = _project_root / "data" / "classiflow.db"
DB_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

print(f"DB path : {_db_path}")

engine = create_async_engine(DB_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

print("database ready")

DB path : C:\Users\leona\source\repos\Trabajo-Integrador\data\classiflow.db
database ready


## 3 — Node helper

Node 4 takes `sha256: str` (from Node 1) and `text: str` (extracted content).
`make_node4` wires the dependencies; `run_node4` runs a single check.

A **shared** `EmbeddingStore` is created once per notebook session so that
successive calls accumulate the in-memory FAISS index — just like the
production coordinator passes the same store across pipeline runs.

In [3]:
from sqlalchemy.ext.asyncio import AsyncSession

# Shared embedding store — accumulates across all cells below
shared_store = EmbeddingStore()


def make_node4(session: AsyncSession, broadcaster: EventBroadcaster) -> DuplicateControlNode:
    return DuplicateControlNode(
        hash_repo=SqlHashRepository(session),
        audit=AuditService(SqlAuditRepository(session)),
        broadcaster=broadcaster,
        embedding_store=shared_store,
    )


def _sha256(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


async def run_node4(job_id: str, filename: str, text: str) -> object:
    async with session_factory() as session:
        result = await make_node4(session, EventBroadcaster()).run(
            job_id=job_id,
            filename=filename,
            sha256=_sha256(text),
            text=text,
        )
        await session.commit()
    return result


print("helpers ready")

helpers ready


## 4 — New document passes

The first time a unique document is submitted, Node 4 saves its SHA-256 to the
hash store and adds its embedding to the FAISS index.

Expected: `passed=True`, `is_duplicate=False`.

In [4]:
DOCUMENT_A = (
    "El Concejo Municipal de Rosario sanciona la siguiente ordenanza: "
    "Artículo 1º — Apruébase el presupuesto municipal para el ejercicio fiscal "
    "correspondiente al año en curso, conforme al detalle que se adjunta como Anexo I "
    "de la presente norma."
)

result = await run_node4("demo-n4-new", "ordenanza.pdf", DOCUMENT_A)

print("=== New document ===")
print(f"  passed           : {result.passed}")
print(f"  is_duplicate     : {result.is_duplicate}")
print(f"  duplicate_type   : {result.duplicate_type}")
print(f"  similarity_score : {result.similarity_score:.4f}")
print(f"  rejection_reason : {result.rejection_reason}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4719.65it/s]
2026-08-06 12:25:40.535 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-n4-new node=node4_duplicate_control event=passed


=== New document ===
  passed           : True
  is_duplicate     : False
  duplicate_type   : 
  similarity_score : 0.0000
  rejection_reason : 


## 5 — Exact duplicate rejected

Submitting the exact same document a second time (same bytes → same SHA-256)
triggers the hash-store check before the embedding index is even consulted.

Expected: `passed=False`, `duplicate_type="exact"`, `similarity_score=1.0`.

In [5]:
result = await run_node4("demo-n4-exact", "ordenanza_copy.pdf", DOCUMENT_A)

print("=== Exact duplicate ===")
print(f"  passed           : {result.passed}")
print(f"  is_duplicate     : {result.is_duplicate}")
print(f"  duplicate_type   : {result.duplicate_type}")
print(f"  similarity_score : {result.similarity_score:.4f}")
print(f"  rejection_reason : {result.rejection_reason}")

2026-08-06 12:25:47.858 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-n4-exact node=node4_duplicate_control event=failed


=== Exact duplicate ===
  passed           : False
  is_duplicate     : True
  duplicate_type   : exact
  similarity_score : 1.0000
  rejection_reason : Exact duplicate: SHA-256 already in store


## 6 — Semantic (near) duplicate rejected

A slightly reworded version of Document A has a different SHA-256 (passes the
hash check) but high cosine similarity in the embedding space — rejected as a
near-duplicate.

Expected: `passed=False`, `duplicate_type="semantic"`, `similarity_score ≥ 0.85`.

In [6]:
DOCUMENT_A_PARAPHRASE = (
    "El Concejo Municipal de la ciudad de Rosario aprueba la siguiente ordenanza: "
    "Artículo 1° — Se aprueba el presupuesto municipal para el año fiscal en curso, "
    "de acuerdo con el detalle incorporado como Anexo I a la presente norma."
)

result = await run_node4("demo-n4-semantic", "ordenanza_v2.pdf", DOCUMENT_A_PARAPHRASE)

print("=== Semantic near-duplicate ===")
print(f"  passed           : {result.passed}")
print(f"  is_duplicate     : {result.is_duplicate}")
print(f"  duplicate_type   : {result.duplicate_type}")
print(f"  similarity_score : {result.similarity_score:.4f}")
print(f"  rejection_reason : {result.rejection_reason}")

2026-08-06 12:25:55.070 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-n4-semantic node=node4_duplicate_control event=failed


=== Semantic near-duplicate ===
  passed           : False
  is_duplicate     : True
  duplicate_type   : semantic
  similarity_score : 0.9326
  rejection_reason : Near-duplicate: cosine similarity 0.933


## 7 — Unrelated document passes

A document on a completely different topic has low cosine similarity and a
unique SHA-256 — it passes and is added to both the hash store and FAISS index.

Expected: `passed=True`, `is_duplicate=False`, `similarity_score < 0.85`.

In [7]:
DOCUMENT_B = (
    "Informe técnico sobre el estado de la red de agua potable del distrito norte. "
    "Se detectaron filtraciones en los tramos comprendidos entre las calles Córdoba "
    "y Corrientes. Se recomienda el reemplazo urgente de 200 metros de cañería."
)

result = await run_node4("demo-n4-unrelated", "informe_agua.pdf", DOCUMENT_B)

print("=== Unrelated document ===")
print(f"  passed           : {result.passed}")
print(f"  is_duplicate     : {result.is_duplicate}")
print(f"  duplicate_type   : {result.duplicate_type}")
print(f"  similarity_score : {result.similarity_score:.4f}")
print(f"  rejection_reason : {result.rejection_reason}")

2026-08-06 12:26:04.384 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-n4-unrelated node=node4_duplicate_control event=passed


=== Unrelated document ===
  passed           : True
  is_duplicate     : False
  duplicate_type   : 
  similarity_score : 0.0000
  rejection_reason : 


## 8 — Inspect the audit records

Node 4 writes an audit entry for every execution, including rejected duplicates.

In [8]:
job_ids = ["demo-n4-new", "demo-n4-exact", "demo-n4-semantic", "demo-n4-unrelated"]

async with session_factory() as session:
    repo = SqlAuditRepository(session)
    all_records = []
    for jid in job_ids:
        all_records.extend(await repo.list_for_job(jid))

print("=== Audit records ===")
for r in all_records:
    print(f"  node          : {r.node}")
    print(f"  event         : {r.event}")
    print(f"  duration_ms   : {r.duration_ms} ms")
    print(f"  detail        : {r.detail}")
    print()

=== Audit records ===
  node          : node4_duplicate_control
  event         : passed
  duration_ms   : 4452 ms
  detail        : {'filename': 'ordenanza.pdf', 'sha256': 'f4aef7ff5e4e57575485d18813af22a9462e625d1c42b801f082f58a6ebd5ea4', 'is_duplicate': False, 'duplicate_type': '', 'similarity_score': 0.0, 'passed': True, 'rejection_reason': ''}

  node          : node4_duplicate_control
  event         : failed
  duration_ms   : 0 ms
  detail        : {'filename': 'ordenanza_copy.pdf', 'sha256': 'f4aef7ff5e4e57575485d18813af22a9462e625d1c42b801f082f58a6ebd5ea4', 'is_duplicate': True, 'duplicate_type': 'exact', 'similarity_score': 1.0, 'passed': False, 'rejection_reason': 'Exact duplicate: SHA-256 already in store'}

  node          : node4_duplicate_control
  event         : failed
  duration_ms   : 31 ms
  detail        : {'filename': 'ordenanza_v2.pdf', 'sha256': '9eec3fae460444c75a7d43f41070a4e888799a42f90430eefc6beaf138a2b2c3', 'is_duplicate': True, 'duplicate_type': 'semantic'

## 9 — Observe SSE events in real time

Node 4 emits `STARTED` then `PASSED`/`FAILED` on every execution.
Subscribe before calling the node to capture both events.

In [9]:
DOCUMENT_C = (
    "Decreto municipal número 1234 del año en curso. Se designa como directora "
    "del área de Cultura y Educación a la señora María López, con efectos a partir "
    "de la fecha de publicación del presente decreto en el Boletín Oficial."
)

broadcaster_sse = EventBroadcaster()
events: list[object] = []


async def collect() -> None:
    async for event in broadcaster_sse.subscribe("demo-n4-sse"):
        events.append(event)
        print(f"  SSE → node={event.node}  status={event.status}")


async with session_factory() as session:
    collect_task = asyncio.create_task(collect())
    await asyncio.sleep(0)  # yield so collect() starts subscribing before run() emits

    await make_node4(session, broadcaster_sse).run(
        job_id="demo-n4-sse",
        filename="decreto.pdf",
        sha256=_sha256(DOCUMENT_C),
        text=DOCUMENT_C,
    )
    await broadcaster_sse.close("demo-n4-sse")
    await collect_task
    await session.commit()

print(f"\ncollected {len(events)} events")

2026-08-06 12:26:14.785 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-n4-sse node=node4_duplicate_control event=passed


  SSE → node=node4_duplicate_control  status=started
  SSE → node=node4_duplicate_control  status=passed

collected 2 events


## 10 — Cleanup

In [10]:
await engine.dispose()
print("engine disposed — database file is now unlocked")

engine disposed — database file is now unlocked
